## Ball and floor

Let a ball of mass $m$ be droped from height $h$. Let gravity be $-g\hat{k}$. The floor-ball collition co-efficient is $\epsilon$.

The ball experiences a gravitational force, 
$$
    \boldsymbol{f} = m \boldsymbol{g},
$$
which can be broken into two ODE,
$$
    \frac{d\boldsymbol{x}}{dt} = \boldsymbol{v},\\
    \frac{d\boldsymbol{v}}{dt} = \frac{\boldsymbol{f}}{m}
$$
The momentum conservation upon bouncing is according to collition co-efficient (elasticity). $[\boldsymbol{v}\rightarrow -\epsilon \boldsymbol{v} ]$


In [14]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Ellipse
from IPython.display import HTML, display
import imageio_ffmpeg
from matplotlib.animation import FFMpegWriter

In [15]:
dt = 0.001
epsilon = 0.8
t = np.arange(0, 5 + dt, dt)
v_0 = [0.1, 0, 0]

In [16]:
# Force defination
def force(m):
    return np.array([0.0, 0.0, -m * 9.81])


def floor_ball_collision(m, h, t, v_0=[0, 0, 0]):
    
    x = np.zeros((len(t), 3))
    v = np.zeros_like(x)
    x[0] = [0, 0, h]
    v[0] = v_0

    for i in range(len(t) - 1):
        # Implicit Euler; gravity is constant, so no solver is needed.
        v[i + 1] = v[i] + dt * force(m) / m
        x[i + 1] = x[i] + dt * v[i + 1]

        # Resolve the floor collision at the end of the time step.
        if x[i + 1, 2] <= 0:
            x[i + 1, 2] = 0
            v[i + 1, 2] = -epsilon * min(v[i + 1, 2], 0)
        if x[i, 2] == 0 and v[i, 2] == 0:
            x[i + 1] = x[i]
            v[i + 1] = v[i]

    return x, v, t


In [17]:
# x-z animation: play/pause and drag the time slider below the plot.
x, v, t = floor_ball_collision(m=1, h=2, t=t, v_0=v_0)

with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 11}):
    fig, ax = plt.subplots(figsize=(9, 6), facecolor="#080e1c")
    fig.subplots_adjust(left=0.12, right=0.94, bottom=0.14, top=0.81)
    ax.set_facecolor("#0c1629")
    height = max(float(x[:, 2].max()), 0.5)
    width = max(0.65 * height, 0.8)
    ax.set(xlim=(-width, width), ylim=(-0.12 * height, 1.18 * height),
           xlabel="x [m]", ylabel="z [m]")
    ax.set_aspect("equal", adjustable="box")
    ax.tick_params(colors="#879bb5", length=0, pad=8)
    ax.xaxis.label.set_color("#b7c9df")
    ax.yaxis.label.set_color("#b7c9df")
    ax.grid(color="#6383a5", alpha=0.12, linewidth=0.7)
    for spine in ax.spines.values():
        spine.set_visible(False)

    fig.text(0.12, 0.93, "GRAVITY / IN MOTION", color="#6ce6ef",
             fontsize=10, weight="bold")
    fig.text(0.12, 0.865, "A ball and a floor - Newton's laws.",
             color="#edf5ff", fontsize=22, weight="bold")
    ax.axvline(0, color="#6ce6ef", alpha=0.12, linestyle="--")
    ax.axhspan(-0.12 * height, 0, color="#14273e")
    for lw, alpha in [(12, 0.04), (6, 0.09), (1.5, 0.9)]:
        ax.axhline(0, color="#6ce6ef", linewidth=lw, alpha=alpha)

    shadow = Ellipse((0, -0.025 * height), 0.2 * height, 0.025 * height,
                     facecolor="#6ce6ef", edgecolor="none", alpha=0.2)
    ax.add_patch(shadow)
    trail, = ax.plot([], [], color="#6ce6ef", lw=3, alpha=0.4,
                     solid_capstyle="round")
    halo = ax.scatter([], [], s=1100, color="#ff85c8", alpha=0.06, zorder=4)
    glow = ax.scatter([], [], s=500, color="#ff85c8", alpha=0.15, zorder=5)
    ball = ax.scatter([], [], s=180, color="#ffd0e9", edgecolor="#ff85c8",
                      linewidth=2, zorder=6)
    highlight = ax.scatter([], [], s=25, color="white", alpha=0.9, zorder=7)
    readout = ax.text(0.035, 0.95, "", transform=ax.transAxes, va="top",
                      color="#d3e5f8", fontsize=11, family="monospace",
                      bbox=dict(boxstyle="round,pad=0.65", facecolor="#080e1c",
                                edgecolor="#23384f", alpha=0.9))
    fig.text(0.12, 0.035, "IMPLICIT EULER   /   RESTITUTION 0.8   /   x = 0",
             color="#879bb5", fontsize=9)

    # Sample at about 30 fps; the simulation still uses its original 1 ms step.
    frames = np.unique(np.r_[np.searchsorted(t, np.arange(0, t[-1], 1 / 30)),
                             len(t) - 1])

    def update(i):
        position = x[i, [0, 2]]
        for artist in (halo, glow, ball):
            artist.set_offsets(position.reshape(1, 2))
        highlight.set_offsets((position + [-0.012 * height, 0.018 * height]).reshape(1, 2))
        start = np.searchsorted(t, max(0, t[i] - 0.22))
        trail.set_data(x[start:i + 1, 0], x[start:i + 1, 2])
        proximity = 1 - min(x[i, 2] / height, 1)
        shadow.width = height * (0.10 + 0.15 * proximity)
        shadow.set_alpha(0.08 + 0.3 * proximity)
        readout.set_text(f"t   {t[i]:5.2f} s\nz   {x[i, 2]:5.2f} m\nv_z {v[i, 2]:+5.2f} m/s")
        return trail, halo, glow, ball, highlight, shadow, readout

    animation = FuncAnimation(fig, update, frames=frames, interval=1000 / 30,
                               blit=False, repeat=True)
    # Embedded frames work without ffmpeg and provide a draggable time slider.
    player = animation.to_jshtml(default_mode="loop")
    plt.close(fig)

display(HTML(player))


In [18]:
# Save the interactive HTML player and MP4 video (30 fps).
import imageio_ffmpeg
from matplotlib.animation import FFMpegWriter
from pathlib import Path

Path("bouncing_ball.html").write_text(player, encoding="utf-8")

with plt.rc_context({"animation.ffmpeg_path": imageio_ffmpeg.get_ffmpeg_exe()}):
    animation.save("bouncing_ball.mp4", writer=FFMpegWriter(fps=30), dpi=150)

print("Saved bouncing_ball.html and bouncing_ball.mp4")


Saved bouncing_ball.mp4
